# New Projects

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from bs4 import BeautifulSoup
import re
import requests

## **Geting All Link OR URI**

In [ ]:
from urllib.parse import urljoin
def get_category_links(base_url):
    """Fetches the category links from the Books to Scrape website."""
    try:
        response = requests.get(base_url, timeout=10)
        print(f"""
        Our Response is {response}
        Type Of Response is {type(response)}
        Response Status : {response.status_code}
        """)

        response.raise_for_status()
        webpage = response.text
        soup = BeautifulSoup(webpage, 'html.parser')

        # Getting all category links
        category_ul = soup.find('ul', class_='nav-list')

        extracted_links = []
        if category_ul:
            # Find all 'a' tags inside the category_ul
            for a_tag in category_ul.find_all('a'):
                link_suffix = a_tag.get('href')
                if link_suffix: # Ensure the href attribute exists
                    # Construct the absolute URL using urljoin for robustness
                    full_link = urljoin(base_url, link_suffix)
                    extracted_links.append(full_link)
        else:
            print("Could not find the category navigation list.")
            return []

        return extracted_links

    except requests.exceptions.HTTPError as http_err:
        print(f"HTTP error occurred: {http_err}") # Specific HTTP error
    except requests.exceptions.ConnectionError as conn_err:
        print(f"Connection error occurred: {conn_err}") # Specific connection error
    except requests.exceptions.Timeout as timeout_err:
        print(f"Timeout error occurred: {timeout_err}") # Specific timeout error
    except requests.exceptions.RequestException as req_err:
        print(f"An unexpected request error occurred: {req_err}") # Catch all other request errors
    except Exception as e:
        print(f"An unexpected error occurred: {e}") # Catch other general exceptions
    return [] # Return an empty list in case of any error

# Main execution block
url = 'https://books.toscrape.com/' # Use the base URL for urljoin

category_links = get_category_links(url)

if category_links:
    print("Successfully extracted category links:")
    # for link in category_links:
    #     print(link)
else:
    print("No category links were extracted due to an error or missing elements.")

## **Getting ALL Data From webpage**

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin
import re

def scrape_all_books(category_urls):
    all_books = []

    # Simple map to convert textual ratings to numbers
    rating_map = {
        'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5
    }

    for cat_url in category_urls:
        current_page_url = cat_url
        print(f"Scraping category: {current_page_url}")

        while current_page_url:
            try:
                response = requests.get(current_page_url, timeout=10)
                if response.status_code != 200:
                    break

                soup = BeautifulSoup(response.text, 'html.parser')

                # Find all book containers on the page
                books = soup.find_all('article', class_='product_pod')

                for book in books:
                    # 1. Title (found inside the h3 -> a tag 'title' attribute)
                    title_tag = book.find('h3').find('a')
                    title = title_tag.get('title') if title_tag else None

                    # 2. Price
                    price_tag = book.find('p', class_='price_color')
                    price = price_tag.text if price_tag else None

                    # 3. Availability
                    availability_tag = book.find('p', class_='instock availability')
                    availability = availability_tag.text.strip() if availability_tag else None

                    rating_tag = book.find('p', class_=re.compile(r'star-rating\s+'))
                    rating_numeric = None
                    if rating_tag:
                        rating_class = rating_tag.get('class')[1]
                        rating_numeric = rating_map.get(rating_class, None)

                    # 5. Book URL
                    book_rel_link = title_tag.get('href') if title_tag else ""
                    book_full_url = urljoin(current_page_url, book_rel_link)

                    all_books.append({
                        'Title': title,
                        'Price': price,
                        'Rating': rating_numeric,
                        'Availability': availability,
                        'URL': book_full_url
                    })

                next_button = soup.find('li', class_='next')
                if next_button:
                    next_rel_url = next_button.find('a').get('href')
                    current_page_url = urljoin(current_page_url, next_rel_url)
                else:
                    current_page_url = None

            except Exception as e:
                print(f"Error scraping {current_page_url}: {e}")
                break

    df = pd.DataFrame(all_books)
    return df


books_df = scrape_all_books(category_links)

print(f"\nTotal books scraped: {len(books_df)}")
books_df.head()

In [ ]:
books_df

## Data Analysis and Visualization

In [ ]:
# Display basic DataFrame information
print("DataFrame Info:")
books_df.info()

print("\nDescriptive Statistics:")
display(books_df.describe(include='all'))

print("\nMissing Values:")
display(books_df.isnull().sum())

### Price Analysis

In [ ]:
# Clean and convert 'Price' to numeric
# Remove currency symbol and convert to float
books_df['Price_Numeric'] = books_df['Price'].str.replace('Â£', '').astype(float)

print("\nPrice Statistics:")
display(books_df['Price_Numeric'].describe())

# Visualize Price Distribution
plt.figure(figsize=(10, 6))
sns.histplot(books_df['Price_Numeric'], bins=30, kde=True)
plt.title('Distribution of Book Prices')
plt.xlabel('Price (£)')
plt.ylabel('Number of Books')
plt.grid(axis='y', alpha=0.75)
plt.show()

plt.figure(figsize=(10, 6))
sns.boxplot(y=books_df['Price_Numeric'])
plt.title('Box Plot of Book Prices')
plt.ylabel('Price (£)')
plt.grid(axis='y', alpha=0.75)
plt.show()

### Rating Analysis

In [ ]:
print("\nRating Distribution:")
display(books_df['Rating'].value_counts().sort_index())

# Visualize Rating Distribution
plt.figure(figsize=(8, 5))
sns.countplot(x='Rating', data=books_df, palette='viridis')
plt.title('Distribution of Book Ratings')
plt.xlabel('Rating (1-5 Stars)')
plt.ylabel('Number of Books')
plt.grid(axis='y', alpha=0.75)
plt.show()

### Availability Analysis

In [ ]:
print("\nAvailability Distribution:")
display(books_df['Availability'].value_counts())

# Visualize Availability Distribution
plt.figure(figsize=(8, 5))
sns.countplot(x='Availability', data=books_df, palette='pastel')
plt.title('Distribution of Book Availability')
plt.xlabel('Availability Status')
plt.ylabel('Number of Books')
plt.grid(axis='y', alpha=0.75)
plt.show()

### Relationship between Price and Rating

In [ ]:
# Box plot of Price by Rating
plt.figure(figsize=(10, 6))
sns.boxplot(x='Rating', y='Price_Numeric', data=books_df, palette='coolwarm')
plt.title('Book Prices by Rating')
plt.xlabel('Rating (1-5 Stars)')
plt.ylabel('Price (£)')
plt.grid(axis='y', alpha=0.75)
plt.show()